In [1]:
import re
from pathlib import Path
import random
from collections import Counter
import math

### Data Loading and Pre-processing

In [2]:
data_root = Path("../inputs/trec06p-ai201")

In [3]:
def dataset_split(X, y, test_size=0.3):
    combined = list(zip(X, y))
    random.shuffle(combined)
    split = int(len(combined) * (1 - test_size))
    X_train, y_train = zip(*combined[:split])
    X_test, y_test = zip(*combined[split:])
    
    return list(X_train), list(X_test), list(y_train), list(y_test)

def create_dataset(data_root):
    with open(f"{data_root}/labels", "r") as f:
        labels, paths = zip(*(line.strip().split(maxsplit=1) for line in f))

    y = list(labels)
    paths = list(paths)

    X = []

    for path in paths:
        file = data_root.joinpath(path.replace("../", ""))
        with open(file, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
            X.append(content)

    return X, y

### Train-Test split 

In [4]:
X,y = create_dataset(data_root)

X_train, X_test, y_train, y_test = dataset_split(X, y, test_size=0.3)

In [5]:
def word_level_tokenize(text):
    # Word = alphabetic sequence, followed by space/comma/period
    pattern = r'[a-zA-Z]+'
    #pattern = r'\b[a-zA-Z]+(?=[\s,\.])'
    tokens = re.findall(pattern, text.lower())
    return [w for w in tokens]

In [6]:
vocab = set()
word_counts = {
    "spam": Counter(),
    "ham": Counter()
}

doc_counts = Counter(y_train)

for text, label in zip(X_train, y_train):
    words = set(word_level_tokenize(text))      
    vocab.update(words)
    word_counts[label].update(words)

V = sorted(vocab)


In [7]:
D_spam = doc_counts["spam"]
D_ham = doc_counts["ham"]

print("D_spam:", D_spam)
print("D_ham:", D_ham)

D_spam: 17478
D_ham: 8997


In [8]:
total_docs = D_spam + D_ham

log_prior = {
    "spam": math.log(D_spam / total_docs),
    "ham": math.log(D_ham / total_docs)
}

P(w∣spam)=Dspam​+λ∣V∣cspam​(w)+λ​P(w∣ham)=Dham​+λ∣V∣cham​(w)+λ​

In [9]:
lmbda = 1
log_likelihood = {
    "spam": {},
    "ham": {}
}

log_prior_spam_denominator = D_spam + lmbda * len(V)
log_prior_ham_denominator = D_ham + lmbda * len(V)

for word in V:
    count_spam = word_counts["spam"][word]
    count_ham = word_counts["ham"][word]

    log_likelihood_spam = math.log((count_spam + lmbda) / log_prior_spam_denominator)
    log_likelihood_ham = math.log((count_ham + lmbda) / log_prior_ham_denominator)

    log_likelihood["spam"][word] = log_likelihood_spam
    log_likelihood["ham"][word] = log_likelihood_ham

scorespam​=logP(spam)+∑logP(w∣spam)
scoreham​=logP(ham)+∑logP(w∣ham)

In [10]:
def predict(text, log_prior, log_likelihood):
    tokens = set(word_level_tokenize(text))   # UNIQUE words (presence model)

    spam_score = log_prior["spam"]
    ham_score  = log_prior["ham"]

    ll_spam = log_likelihood["spam"]
    ll_ham  = log_likelihood["ham"]

    for w in tokens:
        if w in ll_spam:   # w is in training vocab
            spam_score += ll_spam[w]
            ham_score  += ll_ham[w]

    return "spam" if spam_score > ham_score else "ham"

In [11]:
def evaluate(X_test, y_test,
             log_prior,
             log_likelihood):
    TP = FP = FN = TN = 0

    for text, true_label in zip(X_test, y_test):
        pred = predict(
            text,
            log_prior,
            log_likelihood
        )

        if pred == "spam" and true_label == "spam":
            TP += 1
        elif pred == "spam" and true_label == "ham":
            FP += 1
        elif pred == "ham" and true_label == "spam":
            FN += 1
        elif pred == "ham" and true_label == "ham":
            TN += 1

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1_score  = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "TP": TP, "FP": FP, "FN": FN, "TN": TN,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score
    }

In [12]:
results = evaluate(
    X_test, y_test,
    log_prior,
    log_likelihood
)

print(results)
print("Precision:", results["precision"])
print("Recall:", results["recall"])
print("f1_score:", results["f1_score"])

{'TP': 7420, 'FP': 394, 'FN': 14, 'TN': 3519, 'precision': 0.9495776810852317, 'recall': 0.9981167608286252, 'f1_score': 0.9732423924449108}
Precision: 0.9495776810852317
Recall: 0.9981167608286252
f1_score: 0.9732423924449108


In [13]:
len(V)

1355124